# HMM Workflow - Synthetic link generation for 965K

## This Notebook Covers
- Generating the synthetic links for full dataset of 965K complaints
- Inspecting and doing sanity checks of the generated sequences

## Libraries Used
- **Python:** pandas, numpy, matplotlib, seaborn, tqdm

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('data/cfpb_complaints_target_creation.csv')
df['Date received'] = pd.to_datetime(df['Date received'])

print(f"Dataset loaded: {len(df):,} complaints")
print(f"Date range: {df['Date received'].min().date()} to {df['Date received'].max().date()}")

Dataset loaded: 1,399,218 complaints
Date range: 2022-02-04 to 2026-01-15


In [3]:
# Using only temporal training/OOT period (Feb 2024 - Nov 2025)
df_sequences = df[
    (df['Date received'] >= '2024-02-01') & 
    (df['Date received'] < '2025-12-01')
].copy()

print(f"\nFiltered to sequence period: {len(df_sequences):,} complaints")
print(f"Period: {df_sequences['Date received'].min().date()} to {df_sequences['Date received'].max().date()}")


Filtered to sequence period: 965,226 complaints
Period: 2024-02-01 to 2025-11-30


In [4]:
df_sequences.head(2)

,Date received,Product,Sub-product,Issue,Sub-issue,Company,State,ZIP code,Submitted via,Company response to consumer,Timely response?,Consumer disputed?,narrative_clean,narrative_for_lda,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,dominant_topic,topic_granular,topic_category,churn_risk_tier,resolution_tier,year_month
0,2025-01-20,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,Web,Closed with non-monetary relief,Yes,NaN,i am writing to have the following information...,writing have the following information removed...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,23,T23_credit-report_deleted-credit,Identity Theft / Fraud,Moderate,1,2025-01
1,2024-07-03,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Experian Information Solutions Inc.,FL,32824,Web,Closed with non-monetary relief,Yes,NaN,i am a victim of identity theft. please delete...,victim identity theft. please delete remove th...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,5,T05_credit-report_credit-reports,Credit Reporting Errors,Minimal,1,2024-07


In [5]:
SAME_COMPANY = True          # Required
SAME_TOPIC = True            # Required (topic_category)
MIN_DAYS_GAP = 7             # Minimum days between complaints
MAX_DAYS_GAP = 60            # Maximum days between complaints
SAME_STATE = True            # Required (customer unlikely to move in 2 months)
MAX_CHAIN_LENGTH = 4         # Maximum events in a sequence

In [6]:
df_sequences = df_sequences.sort_values('Date received').reset_index(drop=True)
df_sequences['complaint_idx'] = df_sequences.index

TRIAL_SIZE = min(1000000, len(df_sequences))

df_trial = df_sequences.iloc[:TRIAL_SIZE].copy().reset_index(drop=True)

print(f"Total complaints available : {len(df_sequences):,}")
print(f"Trial subset size          : {len(df_trial):,}")

arr_dates = df_trial['Date received'].values.astype('int64')
arr_company = df_trial['Company'].values                              
arr_topic_cat = df_trial['topic_category'].values                       
arr_topic_gran = df_trial['topic_granular'].values                        
arr_state = df_trial['State'].values                                 
arr_zip3 = df_trial['ZIP code'].astype(str).str[:3].values
arr_tier = df_trial['resolution_tier'].values.astype(float)
arr_sent = (
    df_trial['sentiment_compound'].values.astype(float)
    if 'sentiment_compound' in df_trial.columns
    else np.zeros(TRIAL_SIZE)
)
arr_cidx = df_trial['complaint_idx'].values

NS         = 1_000_000_000 * 86_400   
min_gap_ns = MIN_DAYS_GAP * NS
max_gap_ns = MAX_DAYS_GAP * NS
range_ns   = (MAX_DAYS_GAP - MIN_DAYS_GAP) * NS

def find_candidate_links_fast(seed_i, future_indices):
    """
    seed_i         : positional index within df_trial
    future_indices : numpy array of candidate positional indices
                     already narrowed to date window via searchsorted
    Returns        : (best_positional_index, score) or None
    """
    if len(future_indices) == 0:
        return None

    cur_date      = arr_dates[seed_i]
    cur_company   = arr_company[seed_i]
    cur_topic_cat = arr_topic_cat[seed_i]
    cur_topic_gran= arr_topic_gran[seed_i]
    cur_state     = arr_state[seed_i]
    cur_zip3      = arr_zip3[seed_i]
    cur_tier      = arr_tier[seed_i]
    cur_sent      = arr_sent[seed_i]

    f = future_indices

    # Hard filters
    f = f[arr_company[f]    == cur_company]      
    f = f[arr_topic_cat[f]  == cur_topic_cat]     
    f = f[arr_topic_gran[f] == cur_topic_gran]   
    if pd.notna(cur_state):
        f = f[arr_state[f]  == cur_state]         
    f = f[arr_zip3[f]       == cur_zip3] 

    # Escalation: tier must be same or higher
    f = f[arr_tier[f] >= cur_tier]

    # Sentiment: must worsen or stay same
    f = f[arr_sent[f] <= cur_sent]

    if len(f) == 0:
        return None

    # Scoring — tiebreaker between valid candidates only
    # Time proximity (closer to MIN_DAYS_GAP implies more likely same customer)
    time_score = np.clip(
        1.0 - (arr_dates[f] - cur_date - min_gap_ns) / range_ns,
        0.0, 1.0
    ) * 0.6 

    tier_score = np.where(arr_tier[f] > cur_tier, 0.3, 0.1)

    scores = time_score + tier_score + 0.1

    best = scores.argmax()
    return f[best], scores[best]

print(f"\nGenerating sequences for {TRIAL_SIZE:,} complaints...")
print(f"Linking criteria:")
print(f"  Same company            : True")
print(f"  Same topic_category     : True")
print(f"  Same topic_granular     : True")
print(f"  Same state              : True")
print(f"  Same ZIP prefix (3-dig) : True")
print(f"  Tier escalation         : Hard filter (>= current tier)")
print(f"  Sentiment               : Hard filter (<= current sentiment)")
print(f"  Days gap                : {MIN_DAYS_GAP}–{MAX_DAYS_GAP} days")
print(f"  Max chain length        : {MAX_CHAIN_LENGTH}\n")

sequences   = []
sequence_id = 0
used_mask   = np.zeros(TRIAL_SIZE, dtype=bool)

for seed_idx in tqdm(range(TRIAL_SIZE), desc="Building sequences"):

    if used_mask[seed_idx]:
        continue

    chain       = [seed_idx]
    current_idx = seed_idx

    for _ in range(MAX_CHAIN_LENGTH - 1):

        cur_date = arr_dates[current_idx]

        # Binary search into sorted date array
        lo = np.searchsorted(arr_dates, cur_date + min_gap_ns, side='right')
        hi = np.searchsorted(arr_dates, cur_date + max_gap_ns, side='right')
        hi = min(hi, TRIAL_SIZE)

        if lo >= hi:
            break

        future_indices = np.arange(lo, hi)
        future_indices = future_indices[~used_mask[future_indices]]

        result = find_candidate_links_fast(current_idx, future_indices)
        if result is None:
            break

        next_idx = int(result[0])
        chain.append(next_idx)
        used_mask[next_idx] = True
        current_idx = next_idx

    # Record all complaints in this chain
    seq_len = len(chain)
    for pos, idx in enumerate(chain):
        sequences.append({
            'sequence_id':          sequence_id,
            'complaint_idx':        arr_cidx[idx],
            'position_in_sequence': pos,
            'sequence_length':      seq_len,
        })
        used_mask[idx] = True

    sequence_id += 1

Total complaints available : 965,226
Trial subset size          : 965,226

Generating sequences for 965,226 complaints...
Linking criteria:
  Same company            : True
  Same topic_category     : True
  Same topic_granular     : True
  Same state              : True
  Same ZIP prefix (3-dig) : True
  Tier escalation         : Hard filter (>= current tier)
  Sentiment               : Hard filter (<= current sentiment)
  Days gap                : 7–60 days
  Max chain length        : 4



Building sequences: 100%|█████████████████████████████████████████████████████| 965226/965226 [37:21<00:00, 430.54it/s]


In [7]:
df_result = pd.DataFrame(sequences)

print(f"\nSequence generation complete")
print(f"Total sequences created : {sequence_id:,}")
print(f"Output rows             : {len(df_result):,}  (should equal {TRIAL_SIZE:,})")
print(f"Unique complaints       : {df_result['complaint_idx'].nunique():,}  (should equal {TRIAL_SIZE:,})")

print(f"\nSequence length distribution:")
print(df_result.groupby('sequence_length')['sequence_id'].nunique().rename('num_sequences'))

assert len(df_result) == TRIAL_SIZE, \
    f"Row count mismatch: {len(df_result)} != {TRIAL_SIZE}"
assert df_result['complaint_idx'].nunique() == TRIAL_SIZE, \
    "Duplicate complaints detected"


Sequence generation complete
Total sequences created : 623,939
Output rows             : 965,226  (should equal 965,226)
Unique complaints       : 965,226  (should equal 965,226)

Sequence length distribution:
sequence_length
1    443543
2     83146
3     33609
4     63641
Name: num_sequences, dtype: int64


In [8]:
df_check = df_trial.merge(
    df_result[['complaint_idx', 'sequence_id', 'position_in_sequence', 'sequence_length']],
    on='complaint_idx',
    how='left'
)

print(f"\nUntagged complaints: {df_check['sequence_id'].isna().sum()}")


Untagged complaints: 0


In [9]:
df_check

,Date received,Product,Sub-product,Issue,Sub-issue,Company,State,ZIP code,Submitted via,Company response to consumer,Timely response?,Consumer disputed?,narrative_clean,narrative_for_lda,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,dominant_topic,topic_granular,topic_category,churn_risk_tier,resolution_tier,year_month,complaint_idx,sequence_id,position_in_sequence,sequence_length
0,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Portfolio Recovery Associates, LLC",GA,30127,Web,Closed with explanation,Yes,NaN,i have been going through troubling times. i h...,have been going through troubling times. haven...,0.9297,0.047,0.831,0.121,0.0,0.0,0.0,1505,253,12,T12_reporting-agency_consumer-reporting,Credit Reporting Errors,Minimal,0,2024-02,0,0,0,1
1,2024-02-01,Credit card,General-purpose credit card or charge card,Closing your account,Company closed your account,SYNCHRONY FINANCIAL,ME,040XX,Web,Closed with explanation,Yes,NaN,i was told that because i always paid my entir...,was told that because always paid entire balan...,-0.7430,0.085,0.877,0.037,0.8,0.4,0.0,681,121,47,T47_loan_payment,Account / Payment Issues,Moderate,0,2024-02,1,1,0,1
2,2024-02-01,Debt collection,I do not know,Attempts to collect debt not owed,Debt is not yours,Kikoff Inc.,CA,90043,Web,Closed with explanation,Yes,NaN,i have request kickoff to submit to me informa...,have request kickoff submit information where ...,-0.1280,0.062,0.885,0.052,0.0,0.0,0.0,266,46,31,T31_letter_information,Account / Payment Issues,Minimal,0,2024-02,2,2,0,1
3,2024-02-01,Debt collection,Payday loan debt,Attempts to collect debt not owed,Debt was paid,"Upstart Holdings, Inc.",PA,187XX,Web,Closed with explanation,Yes,NaN,this company has been hounding me about a debt...,this company has been hounding about debt that...,-0.6124,0.135,0.865,0.000,0.0,0.0,0.0,164,34,10,T10_debt-collector_debt-collection,Debt Collection,Minimal,0,2024-02,3,3,0,1
4,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"EQUIFAX, INC.",IN,46321,Web,Closed with explanation,Yes,NaN,"from what i know, as per fcra 605b, wrongly re...","from what know, per fcra 605b, wrongly reporte...",0.5055,0.089,0.763,0.148,0.0,0.0,0.0,282,47,33,T33_credit-report_credit-bureaus,Identity Theft / Fraud,Minimal,0,2024-02,4,4,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
965221,2025-11-30,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Experian Information Solutions Inc.,GA,30253,Web,Closed with explanation,Yes,NaN,i am a xxxx xxxx xxxx xxxx. i recently discove...,. recently discovered that was listed part the...,0.1156,0.087,0.808,0.105,0.8,0.0,0.0,683,119,36,T36_identity-theft_theft-report,Identity Theft / Fraud,Minimal,0,2025-11,965221,623936,0,1
965222,2025-11-30,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Experian Information Solutions Inc.,FL,33614,Web,Closed with explanation,Yes,NaN,i am very concerned about the items that are b...,very concerned about the items that are being ...,0.1984,0.094,0.783,0.124,0.0,0.0,0.0,276,51,39,T39_credit-report_inaccurate-accounts,Credit Reporting Errors,Minimal,0,2025-11,965222,600523,1,2
965223,2025-11-30,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,"EQUIFAX, INC.",GA,30324,Web,Closed with explanation,Yes,NaN,i am formally disputing the accuracy and owner...,formally disputing the accuracy and ownership ...,0.4754,0

In [10]:
# Inspecting a sample multi-complaint chain
multi_chains = df_check[df_check['sequence_length'] > 1]['sequence_id'].unique()
print(f"Multi-complaint chains: {len(multi_chains):,}")

sample_seq = multi_chains[0]
print(f"\nSample chain (sequence_id={sample_seq}):")
df_check[df_check['sequence_id'] == sample_seq][[
    'position_in_sequence', 'Date received', 'Company',
    'State', 'ZIP code', 'Company response to consumer',
    'resolution_tier', 'sentiment_compound',
    'topic_category', 'topic_granular'
]].sort_values('position_in_sequence')

Multi-complaint chains: 180,396

Sample chain (sequence_id=4):


,position_in_sequence,Date received,Company,State,ZIP code,Company response to consumer,resolution_tier,sentiment_compound,topic_category,topic_granular
4,0,2024-02-01,"EQUIFAX, INC.",IN,46321,Closed with explanation,0,0.5055,Identity Theft / Fraud,T33_credit-report_credit-bureaus
43621,1,2024-03-21,"EQUIFAX, INC.",IN,46307,Closed with non-monetary relief,1,0.2014,Identity Theft / Fraud,T33_credit-report_credit-bureaus


In [11]:
# Sanity checks: to verify all hard criteria hold across every chain
multi = df_check[df_check['sequence_length'] > 1].copy()
multi = multi.sort_values(['sequence_id', 'position_in_sequence'])

issues = []
for seq_id, grp in multi.groupby('sequence_id'):
    grp = grp.sort_values('position_in_sequence')
    
    # Company, state, zip3, topic_granular must be identical
    for col in ['Company', 'State', 'topic_granular']:
        if grp[col].nunique() > 1:
            issues.append((seq_id, f'{col} mismatch: {grp[col].tolist()}'))
    
    # ZIP prefix
    zip3 = grp['ZIP code'].astype(str).str[:3]
    if zip3.nunique() > 1:
        issues.append((seq_id, f'ZIP prefix mismatch: {zip3.tolist()}'))
    
    # Tier must be non-decreasing
    tiers = grp['resolution_tier'].tolist()
    if tiers != sorted(tiers):
        issues.append((seq_id, f'Tier not escalating: {tiers}'))
    
    # Sentiment must be non-increasing
    sents = grp['sentiment_compound'].tolist()
    if sents != sorted(sents, reverse=True):
        issues.append((seq_id, f'Sentiment not worsening: {sents}'))

print(f"Chains with issues : {len(issues)}")
print(f"Total multi-chains : {multi['sequence_id'].nunique()}")
if issues:
    for seq_id, msg in issues[:10]:
        print(f"  seq {seq_id}: {msg}")
else:
    print("All chains pass hard criteria checks")

Chains with issues : 0
Total multi-chains : 180396
All chains pass hard criteria checks


In [12]:
df_result.to_csv('data/cfpb_synthetic_sequences_900k.csv', index=False)
df_check.to_csv('data/cfpb_sequence_mapped_target_variable.csv', index=False)

In [13]:
multi = df_check[df_check['sequence_length'].isin([3, 4])].copy()

multi['Date received'] = pd.to_datetime(multi['Date received'])

print("Mean seed date by chain length:")
print(
    multi[multi['position_in_sequence'] == 0]
    .groupby('sequence_length')['Date received']
    .mean()
)

Mean seed date by chain length:
sequence_length
3   2025-01-19 10:56:10.882799360
4   2024-12-27 20:07:35.932496384
Name: Date received, dtype: datetime64[ns]


In [14]:
multi.head(10)

,Date received,Product,Sub-product,Issue,Sub-issue,Company,State,ZIP code,Submitted via,Company response to consumer,Timely response?,Consumer disputed?,narrative_clean,narrative_for_lda,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,dominant_topic,topic_granular,topic_category,churn_risk_tier,resolution_tier,year_month,complaint_idx,sequence_id,position_in_sequence,sequence_length
5,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"EQUIFAX, INC.",TX,76036,Web,Closed with explanation,Yes,NaN,i respectfully ask that you eliminate this inc...,respectfully ask that you eliminate this incor...,0.5544,0.108,0.692,0.200,0.0,0.0,0.0,204,32,9,T09_credit-report_credit-reporting,Credit Reporting Errors,Minimal,0,2024-02,5,5,0,4
16,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,Experian Information Solutions Inc.,TX,75052,Web,Closed with explanation,Yes,NaN,i am listing accounts below that have been rep...,listing accounts below that have been reported...,0.9601,0.098,0.656,0.247,1.0,0.0,0.0,710,117,21,T21_credit-reporting_efficiency-banking,Credit Reporting Errors,Minimal,0,2024-02,16,16,0,4
18,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,"EQUIFAX, INC.",CA,95843,Web,Closed with non-monetary relief,Yes,NaN,i discovered that some of the information on m...,discovered that some the information credit re...,0.8968,0.020,0.866,0.114,0.0,0.0,0.0,661,126,23,T23_credit-report_deleted-credit,Identity Theft / Fraud,Minimal,1,2024-02,18,18,0,4
23,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,33023,Web,Closed with non-monetary relief,Yes,NaN,i recently submitted a request for an investig...,recently submitted request for investigation t...,0.6486,0.078,0.763,0.159,0.0,0.0,0.0,354,56,37,T37_credit-reporting_fair-credit,Credit Reporting Errors,Minimal,1,2024-02,23,23,0,4
24,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Personal information incorrect,"EQUIFAX, INC.",LA,70072,Web,Closed with non-monetary relief,Yes,NaN,i'm really not sure what happened. i have mail...,' really not sure what happened. have mailed o...,-0.7183,0.193,0.728,0.079,0.0,0.0,0.0,307,52,44,T44_filing-complaint_misleading-information,General Complaints,Minimal,1,2024-02,24,24,0,4
25,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Personal information incorrect,Experian Information Solutions Inc.,OH,43068,Web,Closed with non-monetary relief,Yes,NaN,"i called experian on xx/xx/2023, xx/xx/2023 an...","called experian //2023, // and // have incorre...",0.8779,0.018,0.929,0.053,0.0,0.0,0.0,1767,304,12,T12_reporting-agency_consumer-reporting,Credit Reporting Errors,Minimal,1,2024-02,25,25,0,3
36,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,"EQUIFAX, INC.",CA,91340,Web,Closed with explanation,Yes,NaN,i lodged a complaint with these bureaus a mont...,"lodged complaint with these bureaus month ago,...",-0.9370,0.157,0.779,0.064,1.0,0.0,0.0,840,143,4,T04_balance-balance_balance-owed,General Complaints,Minimal,0,2024-02,36,36,0,4
40,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",MD,206XX,Web,Closed

In [15]:
multi[multi['sequence_id']==24]

,Date received,Product,Sub-product,Issue,Sub-issue,Company,State,ZIP code,Submitted via,Company response to consumer,Timely response?,Consumer disputed?,narrative_clean,narrative_for_lda,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,dominant_topic,topic_granular,topic_category,churn_risk_tier,resolution_tier,year_month,complaint_idx,sequence_id,position_in_sequence,sequence_length
24,2024-02-01,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Personal information incorrect,"EQUIFAX, INC.",LA,70072,Web,Closed with non-monetary relief,Yes,NaN,i'm really not sure what happened. i have mail...,' really not sure what happened. have mailed o...,-0.7183,0.193,0.728,0.079,0.0,0.0,0.0,307,52,44,T44_filing-complaint_misleading-information,General Complaints,Minimal,1,2024-02,24,24,0,4
11108,2024-02-15,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Personal information incorrect,"EQUIFAX, INC.",LA,70094,Web,Closed with non-monetary relief,Yes,NaN,i'm really not sure what happened. i have mail...,' really not sure what happened. have mailed o...,-0.7183,0.189,0.733,0.078,0.0,0.0,0.0,311,53,44,T44_filing-complaint_misleading-information,General Complaints,Minimal,1,2024-02,11108,24,1,4
19587,2024-02-26,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Personal information incorrect,"EQUIFAX, INC.",LA,70094,Web,Closed with non-monetary relief,Yes,NaN,i'm really not sure what happened. i have mail...,' really not sure what happened. have mailed o...,-0.7183,0.193,0.728,0.079,0.0,0.0,0.0,307,52,44,T44_filing-complaint_misleading-information,General Complaints,Minimal,1,2024-02,19587,24,2,4
42159,2024-03-20,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,"EQUIFAX, INC.",LA,700XX,Web,Closed with non-monetary relief,Yes,NaN,i'm really not sure what happened. i have mail...,' really not sure what happened. have mailed o...,-0.7183,0.193,0.728,0.079,0.0,0.0,0.0,307,52,44,T44_filing-complaint_misleading-information,General Complaints,Minimal,1,2024-03,42159,24,3,4


In [16]:
multi[multi['sequence_id']==24].to_csv('synthetic_multi_chain_sample.csv',index=False)